In [ ]:
# =================================
# IMPORT des Bibliothèques
# =================================
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import json
from tqdm import tqdm
from datetime import datetime

In [25]:
# =================================
# Webscrapping
# =================================

# Préparation - input sur le loc en provisoire pour test
loc = input("ville?")
url = f"https://www.agendaculturel.fr/?q={loc}"
html = requests.get(url)
soup = BeautifulSoup(html.text, "html.parser")

In [26]:
MOIS = {
    1: "janvier",
    2: "février",
    3: "mars",
    4: "avril",
    5: "mai",
    6: "juin",
    7: "juillet",
    8: "août",
    9: "septembre",
    10: "octobre",
    11: "novembre",
    12: "décembre",
}

for card in soup.select("div.card-body"):
    data = {}
    # ======================
    # DATE
    badge = card.select_one("span.card-main-badge")
    date_text = ""
    if badge:
        times = badge.find_all("time")
        # Cas : date unique
        if len(times) == 1:
            d = datetime.fromisoformat(times[0]["datetime"])
            texte_badge = badge.get_text(" ", strip=True).lower()

            # Cas : Jusqu'au
            if "jusqu" in texte_badge:
                date_text = f"jusqu'au {d.day} {MOIS[d.month]} {d.year}"

            # Cas : date simple
            else:
                date_text = f"{d.day} {MOIS[d.month]} {d.year}"
        # Cas : Du ... au ...
        elif len(times) == 2:
            d1 = datetime.fromisoformat(times[0]["datetime"])
            d2 = datetime.fromisoformat(times[1]["datetime"])
            # même mois / même année
            if d1.month == d2.month and d1.year == d2.year:
                date_text = (
                    f"du {d1.day} au {d2.day} "
                    f"{MOIS[d2.month]} {d2.year}"
                )
            # mois différents
            else:
                date_text = (
                    f"du {d1.day} {MOIS[d1.month]} {d1.year} "
                    f"au {d2.day} {MOIS[d2.month]} {d2.year}"
                )
    data["date"] = date_text

    # ======================
    # TITRE
    titre = card.select_one("div.h5.card-title [itemprop='name']")
    data["titre"] = titre.get_text(strip=True) if titre else ""

    # ======================
    # LIEU
    lieu = card.select_one('[itemprop="location"] [itemprop="name"]')
    data["lieu"] = lieu.get_text(strip=True) if lieu else ""

    # ======================
    # DESCRIPTION
    desc = card.select_one('[itemprop="description"]')
    data["description"] = desc.get_text(" ", strip=True) if desc else ""

    print(data)

{'date': "jusqu'au 25 juillet 2026", 'titre': 'Festival de Nîmes', 'lieu': 'Arènes de Nimes', 'description': 'Festival de Nîmes 2026, 29e édition d’exception aux Arènes devant 175 000 spectateurs'}
{'date': 'du 13 au 14 juillet 2026', 'titre': 'Fête Nationale Nîmes', 'lieu': 'Divers Lieux', 'description': '14 juillet à Nîmes entre défilé, animations festives et grand bal populaire'}
{'date': 'du 20 au 21 juin 2026', 'titre': 'Fête de la musique à Nimes', 'lieu': '', 'description': 'Fête de la musique 2026 à Nîmes, deux jours de concerts gratuits et scènes vibrantes'}
{'date': 'du 19 au 20 septembre 2026', 'titre': 'Journées du patrimoine Nimes', 'lieu': '', 'description': 'Journées du Patrimoine 2026 à Nîmes entre trésors antiques et patrimoine en danger'}
{'date': '22 mai 2027', 'titre': 'Nuit des musées à Nimes', 'lieu': '', 'description': 'Nuit des musées à Nîmes 2027 visites, concerts et théâtre jusqu’à minuit'}
{'date': 'du 1 au 31 décembre 2026', 'titre': 'Spectacles de noël à Ni